# NLP Encoder — PyTorch
### MedAI Diagnose | CNN + NLP + PEPA

**Architecture:**
```
TF-IDF(500) → Linear(512) → BN1d → ReLU → Dropout(0.3)
            → Linear(256) → BN1d → ReLU → Dropout(0.3)
            → Linear(128) → ReLU
            → f_txt ∈ R^128
```

In [1]:
# Cell 1 — GPU Setup
# Uncomment below ONCE to install, then comment back and restart kernel
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

import torch
import torch.nn as nn
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'Device          : {device}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'Compute cap     : sm_{torch.cuda.get_device_properties(0).major}{torch.cuda.get_device_properties(0).minor}')
    torch.backends.cudnn.benchmark = True

PyTorch version : 2.10.0+cu128
CUDA available  : True
Device          : cuda
GPU             : NVIDIA GeForce RTX 5050 Laptop GPU
VRAM            : 8.5 GB
Compute cap     : sm_120


In [2]:
# Cell 2 — NLP Encoder Class

class NLPEncoder(nn.Module):
    """
    Dense encoder for TF-IDF symptom vectors.

    Input  : (batch, input_dim)  — TF-IDF vector
    Output : (batch, embed_dim)  — symptom embedding f_txt
    """
    def __init__(self, input_dim: int = 500, embed_dim: int = 128):
        super(NLPEncoder, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, embed_dim),
            nn.ReLU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Build and inspect
nlp = NLPEncoder(input_dim=500, embed_dim=128).to(device)
print(nlp)
print(f'\nParams: {sum(p.numel() for p in nlp.parameters()):,}')

NLPEncoder(
  (net): Sequential(
    (0): Linear(in_features=500, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
  )
)

Params: 422,272


In [3]:
# Cell 3 — Quick Test

nlp.eval()
with torch.no_grad():
    x = torch.randn(8, 500).to(device)
    out = nlp(x)

print(f'Input  : {x.shape}')
print(f'Output : {out.shape}')
print(f'Device : {out.device}')
print('\n✓ NLP Encoder ready')

Input  : torch.Size([8, 500])
Output : torch.Size([8, 128])
Device : cuda:0

✓ NLP Encoder ready
